# DataFrame to LaTeX Table Converter

This notebook provides a dedicated tool (`LatexTableFormatter`) to format DataFrames into publication-ready LaTeX tables, with specialized support for:
- Concatenating central values and uncertainties into LaTeX `$val \pm unc$` syntax.
- Automatic uncertainty-based rounding (PDG standard: 2 significant figures for uncertainty with aligned central value decimal precision).
- Fixed decimal precision rounding.
- Auto-escaping LaTeX underscores and customizable tabular wrappers (`caption`, `label`, `column_format`, `resizebox`).

In [5]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from pathlib import Path
import sys
from typing import Optional
from src.core.Validation.LatexTableFormatter import LatexTableFormatter

# Functions

In [6]:
def concatenate_pm_columns(
    df: pd.DataFrame,
    val_col: str,
    unc_col: str,
    new_col: Optional[str] = None,
    precision: Optional[int] = None,
    sig_figs: int = 2,
    drop_original: bool = False,
) -> pd.DataFrame:
    r"""Combines two columns into one formatted as '$val \pm unc$'."""
    return LatexTableFormatter.concatenate_columns(
        df=df,
        val_col=val_col,
        unc_col=unc_col,
        new_col=new_col,
        precision=precision,
        sig_figs=sig_figs,
        drop_original=drop_original,
    )


def dataframe_to_latex(
    df: pd.DataFrame,
    caption: Optional[str] = None,
    label: Optional[str] = None,
    column_format: Optional[str] = None,
    escape_underscores: bool = True,
    resize_box: bool = False,
    index: bool = False,
) -> str:
    """Converts a DataFrame directly into a LaTeX table snippet."""
    return LatexTableFormatter.to_latex(
        df=df,
        caption=caption,
        label=label,
        column_format=column_format,
        escape_underscores=escape_underscores,
        resize_box=resize_box,
        index=index,
    )

# Exemples

## Example A: Synthetic Distance Metrics & Automatic Rounding (PDG Style)


In [4]:
# Create sample DataFrame with diverse magnitudes
sample_df = pd.DataFrame({
    "Feature": ["trig_L2_calo_et", "trig_L2_calo_eta", "Eratio"],
    "Comparison": ["Model V9 vs Model V10", "Model V9 vs Model V10", "Model V9 vs Model V10"],
    "KS_Distance": [0.008336, 0.006995, 0.233209],
    "KS_Uncertainty": [0.005644, 0.006027, 0.008621],
    "Wasserstein": [16.628878, 0.010359, 0.155185],
    "Wasserstein_Uncertainty": [23.797069, 0.012521, 0.005474],
    "JS_Divergence": [0.000370, 0.000194, 0.048428],
    "JS_Uncertainty": [0.000782, 0.000495, 0.003412],
})

display(sample_df)

,Feature,Comparison,KS_Distance,KS_Uncertainty,Wasserstein,Wasserstein_Uncertainty,JS_Divergence,JS_Uncertainty
0,trig_L2_calo_et,Model V9 vs Model V10,0.008336,0.005644,16.628878,23.797069,0.000370,0.000782
1,trig_L2_calo_eta,Model V9 vs Model V10,0.006995,0.006027,0.010359,0.012521,0.000194,0.000495
2,Eratio,Model V9 vs Model V10,0.233209,0.008621,0.155185,0.005474,0.048428,0.003412


## Example B: Merging Multiple Columns with Auto-Rounding (`sig_figs=2`)

In [5]:
# Merge pairs into unified LaTeX columns
pairs = {
    "KS Distance ($D$)": ("KS_Distance", "KS_Uncertainty"),
    "Wasserstein ($W_1$)": ("Wasserstein", "Wasserstein_Uncertainty"),
    "JS Divergence ($D_{\\text{JS}}$)": ("JS_Divergence", "JS_Uncertainty"),
}

# Automatic PDG rounding
formatted_auto = LatexTableFormatter.merge_uncertainty_pairs(
    sample_df,
    pairs=pairs,
    precision=None,
    sig_figs=2,
    drop_original=True,
)

print("=== Preview in Notebook ===")
display(formatted_auto)

print("=== Generated LaTeX Code ===")
latex_code = dataframe_to_latex(
    formatted_auto,
    caption="Statistical Distance Metrics with Automatic Uncertainty Rounding.",
    label="tab:distance_metrics",
)
print(latex_code)

=== Preview in Notebook ===


,Feature,Comparison,KS Distance ($D$),Wasserstein ($W_1$),JS Divergence ($D_{\text{JS}}$)
0,trig_L2_calo_et,Model V9 vs Model V10,$0.0083 \pm 0.0056$,$17 \pm 24$,$0.00037 \pm 0.00078$
1,trig_L2_calo_eta,Model V9 vs Model V10,$0.0070 \pm 0.0060$,$0.010 \pm 0.013$,$0.00019 \pm 0.00049$
2,Eratio,Model V9 vs Model V10,$0.2332 \pm 0.0086$,$0.1552 \pm 0.0055$,$0.0484 \pm 0.0034$


=== Generated LaTeX Code ===
\begin{table}[htbp]
\centering
\caption{Statistical Distance Metrics with Automatic Uncertainty Rounding.}
\label{tab:distance_metrics}
\begin{tabular}{l l l l l}
\toprule
Feature & Comparison & KS Distance ($D$) & Wasserstein ($W_1$) & JS Divergence ($D_{\text{JS}}$) \\
\midrule
trig\_L2\_calo\_et & Model V9 vs Model V10 & $0.0083 \pm 0.0056$ & $17 \pm 24$ & $0.00037 \pm 0.00078$ \\
trig\_L2\_calo\_eta & Model V9 vs Model V10 & $0.0070 \pm 0.0060$ & $0.010 \pm 0.013$ & $0.00019 \pm 0.00049$ \\
Eratio & Model V9 vs Model V10 & $0.2332 \pm 0.0086$ & $0.1552 \pm 0.0055$ & $0.0484 \pm 0.0034$ \\
\bottomrule
\end{tabular}
\end{table}



## Example C: Fixed Decimal Precision Rounding (`precision=4`)

In [6]:
# Merge with fixed precision (4 decimal places)
formatted_fixed = LatexTableFormatter.merge_uncertainty_pairs(
    sample_df,
    pairs=pairs,
    precision=4,
    drop_original=True,
)

display(formatted_fixed)
print(dataframe_to_latex(formatted_fixed))

,Feature,Comparison,KS Distance ($D$),Wasserstein ($W_1$),JS Divergence ($D_{\text{JS}}$)
0,trig_L2_calo_et,Model V9 vs Model V10,$0.0083 \pm 0.0056$,$16.6289 \pm 23.7971$,$0.0004 \pm 0.0008$
1,trig_L2_calo_eta,Model V9 vs Model V10,$0.0070 \pm 0.0060$,$0.0104 \pm 0.0125$,$0.0002 \pm 0.0005$
2,Eratio,Model V9 vs Model V10,$0.2332 \pm 0.0086$,$0.1552 \pm 0.0055$,$0.0484 \pm 0.0034$


\begin{tabular}{l l l l l}
\toprule
Feature & Comparison & KS Distance ($D$) & Wasserstein ($W_1$) & JS Divergence ($D_{\text{JS}}$) \\
\midrule
trig\_L2\_calo\_et & Model V9 vs Model V10 & $0.0083 \pm 0.0056$ & $16.6289 \pm 23.7971$ & $0.0004 \pm 0.0008$ \\
trig\_L2\_calo\_eta & Model V9 vs Model V10 & $0.0070 \pm 0.0060$ & $0.0104 \pm 0.0125$ & $0.0002 \pm 0.0005$ \\
Eratio & Model V9 vs Model V10 & $0.2332 \pm 0.0086$ & $0.1552 \pm 0.0055$ & $0.0484 \pm 0.0034$ \\
\bottomrule
\end{tabular}



# Models

## ModelV9 100Rings id:20261029001623

### et_1_eta_1_operating_points_efficiency

In [8]:
# Example: Scan for a real distance metrics table
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
csv_files = "../notebooks/Tables/multi_model_mean_summary.parquet"

if csv_files:
    # target_csv = csv_files[0]
    # print(f"Loading: {target_csv.name}")
    try:
        real_df = pd.read_csv(csv_files)
    except:
        real_df = pd.read_parquet(csv_files)

    metric_pairs = {
        "PD (Signal Eff) ": ("PD mean [%]", "PD std [%]"),
        "PF (Fake Rate)": ("PF mean [%]", "PF std [%]"),
        "SP": ("SP mean [%]", "SP std [%]"),
    }

    # Select only Feature, Comparison, and metric columns
    keep_cols = ["method"] + [c for pair in metric_pairs.values() for c in pair]
    sub_df = real_df[[c for c in keep_cols if c in real_df.columns]]

    exported_df = LatexTableFormatter.merge_uncertainty_pairs(
        sub_df,
        pairs=metric_pairs,
        sig_figs=2,
        precision=3,
        drop_original=True,
    )

    display(exported_df.head(10))
    print("\n--- Exported LaTeX ---")
    print(dataframe_to_latex(exported_df.head(10)))
else:
    print("No distance metric CSV files discovered.")

real_df

,method,PD (Signal Eff),PF (Fake Rate),SP
0,Model V9 (100 Rings) [Best SP],$81.360 \pm 7.340$,$14.900 \pm 6.040$,$83.200 \pm 6.460$
1,Model V9 (100 Rings) [Hard Cut],$63.900 \pm 23.820$,$8.250 \pm 6.810$,$75.920 \pm 13.680$
2,Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\...,$64.020 \pm 23.780$,$8.220 \pm 6.790$,$76.020 \pm 13.610$
3,Athena Loose Cut,$64.580 \pm 22.780$,$29.070 \pm 14.580$,$66.220 \pm 9.250$
4,Model V9 (50 Rings) [Best SP],$81.540 \pm 6.510$,$15.910 \pm 7.020$,$82.790 \pm 6.410$
5,Model V9 (50 Rings) [Hard Cut],$63.900 \pm 23.820$,$8.780 \pm 7.070$,$75.670 \pm 13.600$
6,Model V9 (50 Rings) [$y_\mathrm{cut}(\langle\m...,$64.150 \pm 23.940$,$8.850 \pm 7.070$,$75.780 \pm 13.670$
7,Model V10 (100 Rings) [Best SP],$81.580 \pm 7.290$,$16.410 \pm 6.990$,$82.560 \pm 6.770$
8,Model V10 (100 Rings) [Hard Cut],$63.900 \pm 23.820$,$8.990 \pm 7.210$,$75.570 \pm 13.560$
9,Model V10 (100 Rings) [$y_\mathrm{cut}(\langle...,$64.170 \pm 23.730$,$9.030 \pm 7.140$,$75.740 \pm 13.460$



--- Exported LaTeX ---
\begin{tabular}{l l l l}
\toprule
method & PD (Signal Eff)  & PF (Fake Rate) & SP \\
\midrule
Model V9 (100 Rings) [Best SP] & $81.360 \pm 7.340$ & $14.900 \pm 6.040$ & $83.200 \pm 6.460$ \\
Model V9 (100 Rings) [Hard Cut] & $63.900 \pm 23.820$ & $8.250 \pm 6.810$ & $75.920 \pm 13.680$ \\
Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\mu\rangle)$] & $64.020 \pm 23.780$ & $8.220 \pm 6.790$ & $76.020 \pm 13.610$ \\
Athena Loose Cut & $64.580 \pm 22.780$ & $29.070 \pm 14.580$ & $66.220 \pm 9.250$ \\
Model V9 (50 Rings) [Best SP] & $81.540 \pm 6.510$ & $15.910 \pm 7.020$ & $82.790 \pm 6.410$ \\
Model V9 (50 Rings) [Hard Cut] & $63.900 \pm 23.820$ & $8.780 \pm 7.070$ & $75.670 \pm 13.600$ \\
Model V9 (50 Rings) [$y_\mathrm{cut}(\langle\mu\rangle)$] & $64.150 \pm 23.940$ & $8.850 \pm 7.070$ & $75.780 \pm 13.670$ \\
Model V10 (100 Rings) [Best SP] & $81.580 \pm 7.290$ & $16.410 \pm 6.990$ & $82.560 \pm 6.770$ \\
Model V10 (100 Rings) [Hard Cut] & $63.900 \pm 23.820$ & $

,config_name,method,PD mean [%],PD std [%],SP mean [%],SP std [%],PF mean [%],PF std [%]
0,ModelV9_HighBatch_100Rings,Model V9 (100 Rings) [Best SP],81.36,7.34,83.20,6.46,14.90,6.04
1,ModelV9_HighBatch_100Rings,Model V9 (100 Rings) [Hard Cut],63.90,23.82,75.92,13.68,8.25,6.81
2,ModelV9_HighBatch_100Rings,Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\...,64.02,23.78,76.02,13.61,8.22,6.79
3,Athena,Athena Loose Cut,64.58,22.78,66.22,9.25,29.07,14.58
4,ModelV9_HighBatch_50Rings,Model V9 (50 Rings) [Best SP],81.54,6.51,82.79,6.41,15.91,7.02
5,ModelV9_HighBatch_50Rings,Model V9 (50 Rings) [Hard Cut],63.90,23.82,75.67,13.60,8.78,7.07
6,ModelV9_HighBatch_50Rings,Model V9 (50 Rings) [$y_\mathrm{cut}(\langle\m...,64.15,23.94,75.78,13.67,8.85,7.07
7,ModelV10_HighBatch_100Rings,Model V10 (100 Rings) [Best SP],81.58,7.29,82.56,6.77,16.41,6.99
8,ModelV10_HighBatch_100Rings,Model V10 (100 Rings) [Hard Cut],63.90,23.82,75.57,13.56,8.99,7.21
9,ModelV10_HighBatch_100Rings,Model V10 (100 Rings) [$y_\mathrm{cut}(\langle...,64.17,23.73,75.74,13.46,9.03,7.14
